# PitWall Model Refinement

Cache-first notebook for inspecting PitWall model data, feature stability, leakage risk, chronological validation, champion artifacts, and optional challenger training. Outputs are intentionally small so the notebook can stay in Git.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'f1_briefing.py').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

DATA_CACHE = ROOT / 'data_cache'
MODEL_ARTIFACTS = ROOT / 'model_artifacts'
SAVED_MODELS = ROOT / 'models' / 'saved_models'
BRIEFINGS = ROOT / 'briefings'
print(ROOT)

## Load Existing Artifacts

This notebook prefers checked-in/cache artifacts and does not redownload upstream data.

In [ ]:
def read_json(path, fallback=None):
    path = Path(path)
    if not path.exists():
        return fallback
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        return {'error': str(exc), 'path': str(path)}

frontend_contract = read_json(DATA_CACHE / 'frontend-contract.json', {})
debug_contract = read_json(DATA_CACHE / 'latest-model-debug.json', {})
model_status = read_json(DATA_CACHE / 'model-status.json', {})
backtest_history = read_json(DATA_CACHE / 'backtest-history.json', {})
cache_manifest = read_json(DATA_CACHE / 'cache_manifest.json', {'entries': {}})
briefing_index = read_json(BRIEFINGS / 'index.json', {'briefings': []})

summary = {
    'top10_rows': len((frontend_contract.get('latest') or {}).get('top10') or []),
    'full_grid_rows': len((frontend_contract.get('latest') or {}).get('full_grid') or []),
    'debug_payloads': len(debug_contract.get('payloads') or []),
    'briefings': len(briefing_index.get('briefings') or []),
    'cache_manifest_entries': len(cache_manifest.get('entries') or {}),
}
summary

## Feature Availability And Missing Values

In [ ]:
import pandas as pd
from pitwall.features.build_features import feature_schema, missing_value_report

feature_path = DATA_CACHE / 'ml_full_race_features.csv'
raw_path = DATA_CACHE / 'ml_full_race_results_raw.csv'
features = pd.read_csv(feature_path) if feature_path.exists() else pd.DataFrame()
raw_results = pd.read_csv(raw_path) if raw_path.exists() else pd.DataFrame()

candidate_features = [c for c in features.columns if c not in {'race_id','season','round','race_name','circuit_id','driver_id','driver_name','constructor','finish_position','is_win','is_podium','is_top10','target_lap_pace'}]
{
    'feature_rows': len(features),
    'raw_rows': len(raw_results),
    'race_groups': int(features['race_id'].nunique()) if 'race_id' in features else 0,
    'schema': feature_schema(candidate_features[:12]),
    'missing_sample': missing_value_report(features, candidate_features[:12]) if not features.empty else {},
}

## Leakage And Chronological Group Checks

In [ ]:
from pitwall.models.validation import validate_training_frame, validate_feature_stage
from pitwall.models.train import chronological_split

training_validation = validate_training_frame(features, candidate_features) if not features.empty else {'ok': False, 'errors': ['no features CSV']}
stage_checks = {stage: validate_feature_stage(stage, candidate_features) for stage in ['pre_weekend', 'post_fp3', 'post_sprint_qualifying', 'post_sprint', 'post_qualifying', 'pre_race']}
split_info = {}
if not features.empty and training_validation['ok']:
    train_df, valid_df, split_info = chronological_split(features)
    split_info = split_info
{'training_validation': training_validation, 'stage_checks': stage_checks, 'split': split_info}

## Champion Model And Metrics

In [ ]:
from pitwall.models.artifacts import artifact_summary, load_model_metadata

bundle_path = SAVED_MODELS / 'f1_hybrid_full_data_bundle.pkl'
meta_path = SAVED_MODELS / 'f1_hybrid_full_data_meta.json'
champion_meta = load_model_metadata(meta_path)
{
    'artifact_summary': artifact_summary(bundle_path, meta_path),
    'metric_keys': sorted((champion_meta.get('metrics') or {}).keys())[:20],
    'validation_split': (champion_meta.get('metrics') or {}).get('validation_split') or champion_meta.get('validation_split'),
}

## Evaluate Existing Predictions

When actual finish positions are available in the feature table, inspect finish and ranking metrics with race grouping.

In [ ]:
from pitwall.models.evaluate import evaluate_finish_predictions, probability_metrics

eval_summary = {'status': 'not_enough_data'}
if not features.empty and {'race_id', 'driver_id', 'finish_position', 'grid_position'}.issubset(features.columns):
    sample = features.dropna(subset=['finish_position']).tail(220).copy()
    if len(sample):
        eval_summary = evaluate_finish_predictions(sample, sample['grid_position'].fillna(sample['grid_position'].median()))
        if 'is_top10' in sample:
            eval_summary['grid_top10_brier_proxy'] = probability_metrics(sample['is_top10'].astype(int), (21 - sample['grid_position'].fillna(20)).clip(0, 20) / 20)
eval_summary

## Feature Importance And Ablation Hooks

This reads saved importance when present. Heavier permutation importance or ablation should be run only when dependencies and runtime budget are available.

In [ ]:
importance = read_json(MODEL_ARTIFACTS / 'feature_importance.json', {})
importance_rows = importance.get('features') or importance.get('feature_importance') or []
pd.DataFrame(importance_rows).head(12) if importance_rows else {'status': 'no saved feature importance artifact'}

## Optional Challenger Training

Set `RUN_CHALLENGER = True` only when the local cache is valid and you intend to spend the runtime. Promotion must still pass backend tests, contract validation, artifact save/load, frontend tests, and build.

In [ ]:
from pitwall.models.train import train_or_load_champion, training_status
from pitwall.models.validation import promotion_gate

RUN_CHALLENGER = False
status = training_status(force=False)
challenger = None
if RUN_CHALLENGER and training_validation.get('ok'):
    challenger = train_or_load_champion(force=True)

promotion_gate(champion_meta.get('metrics'), (challenger or {}).get('metrics') if challenger else None) | {'training_status': status.get('action')}

## Interpreting Results

- Prefer chronological race-group validation over random row splits.
- Treat classifier AUC/Brier as probability diagnostics, not proof of race-ranking quality.
- Promotion should consider finish MAE/RMSE, Spearman, NDCG@3/10, top-3/top-10 recall, winner hit rate, calibration, artifact reload, backend tests, frontend contract validation, frontend tests, and build.
- Predictions are probabilistic; missing, stale, or fallback sources should reduce trust and be visible in source health.